# AlgoSathi quickstart

Load the built index, run a query in three registers, and look inside the retrieval
pipeline. Run `make all` first (or at least `make data && make index`).

> The corpus here is **synthetic**. See `docs/findings.md`.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.utils.config import load_config
from src.inference.rag_pipeline import AlgoSathiRAG

cfg = load_config('configs/default.yaml')
rag = AlgoSathiRAG(cfg)
print('generator backend:', rag.generator.name)

## 1. The knowledge base (Table 1)


In [ ]:
import pandas as pd
from src.data.loaders import load_knowledge_base_df

kb = load_knowledge_base_df(cfg)
print(len(kb), 'chunks |', (kb.language=='cm').sum(), 'code-mixed')
kb.groupby(['topic','language']).size().unstack(fill_value=0)

## 2. Code-switch detection and query branching

The same question in three registers takes three different preprocessing branches.


In [ ]:
from src.preprocessing.query_pipeline import preprocess_query

queries = [
    'what is the time complexity of merge sort',
    'merge sort ko complexity kasari nikalne',
    'yo merge sort kasari kaam garcha hola bhanera bujhna sajilo cha',
]
for q in queries:
    p = preprocess_query(q, rag.detector, cfg)
    print(f'{p.query_type:18s} expanded={p.expanded}  '
          f"profile={ {k: round(v,2) for k,v in p.language_profile.items()} }")
    print('   encodes:', p.encode_text[:110], '\n')

## 3. Inside one retrieval call

Dense score, BM25 score, fusion score and re-rank score for every returned chunk.


In [ ]:
hits = rag.retrieve_only('yo dijkstra negative weight ma kina fail huncha')
pd.DataFrame([{'rank': h['rank'], 'chunk_id': h['chunk_id'], 'concept': h['concept'],
               'lang': h['language'], 'dense': round(h['dense_score'],3),
               'bm25': round(h['bm25_score'],2), 'fusion': round(h['fusion_score'],4),
               'rerank': round(h.get('rerank_score',0),3)} for h in hits])

## 4. Ablating the retriever

The switches used to build Table 3 are arguments to a single method.


In [ ]:
from src.evaluation.retrieval_metrics import evaluate_query
from src.data.loaders import load_eval_queries

queries = load_eval_queries(cfg, 'romanized_nepali')[:40]
configs = {'dense_only': dict(use_dense=True, use_bm25=False, rerank=False),
           'bm25_only': dict(use_dense=False, use_bm25=True, rerank=False),
           'hybrid+rerank': dict(use_dense=True, use_bm25=True, rerank=True)}
rows = []
for name, kwargs in configs.items():
    scores = []
    for q in queries:
        hits = rag.retriever.retrieve(q['query'], top_k=10, **kwargs)
        scores.append(evaluate_query([h['chunk_id'] for h in hits],
                                     q['relevant_chunk_ids'], ks=(5,10)))
    rows.append({'retriever': name,
                 'P@5': sum(s['P@5'] for s in scores)/len(scores),
                 'MRR': sum(s['MRR'] for s in scores)/len(scores)})
pd.DataFrame(rows).round(3)

## 5. A full grounded answer


In [ ]:
r = rag.ask('binary search ko lower bound bhaneko k ho')
print(r.query_type, '|', round(r.latency_s,2), 's | grounded:', r.grounded)
print(r.answer)

## 6. The statistics, on any scores CSV


In [ ]:
from src.data.loaders import load_study_scores
from src.evaluation.evaluate_learning import analyse_scores

scores = load_study_scores(cfg)          # or load_study_scores(cfg, 'my_scores.csv')
res = analyse_scores(scores)
print({k: res['gain_t_test'][k] for k in ('mean_diff','t','p_value','cohens_d')})
print({k: res['ancova'][k] for k in ('adjusted_difference','ci_low','ci_high','p_value')})